# Workflows Avanzados: Sub-workflows, Reintentos y Producción

Patrones avanzados que hacen los workflows de n8n robustos y listos para producción.

**Contenidos:**
1. Sub-workflow reutilizable `SW_LlamarClaude`
2. Manejo de errores con reintentos y backoff exponencial
3. Rate limiting con estado persistente
4. Checklist de producción automatizado

In [ ]:
import anthropic
import json
import time
from datetime import datetime
from collections import defaultdict

client = anthropic.Anthropic()
print("Cliente Anthropic listo")

## 1. Sub-workflow: SW_LlamarClaude

En n8n los sub-workflows permiten reutilizar lógica entre workflows.
El equivalente Python es una función bien definida con inputs/outputs documentados.

In [ ]:
PRECIOS_MODELOS = {
    "claude-haiku-4-5-20251001": {"input": 0.80, "output": 4.00},
    "claude-sonnet-4-6": {"input": 3.00, "output": 15.00}
}

def sw_llamar_claude(
    prompt: str,
    system: str = None,
    modelo: str = "claude-haiku-4-5-20251001",
    max_tokens: int = 500
) -> dict:
    """
    Equivalente Python del sub-workflow SW_LlamarClaude en n8n.
    
    Inputs esperados (como en el Execute Workflow Trigger de n8n):
      - prompt: str
      - system: str (opcional)
      - modelo: str (opcional, default: claude-haiku-4-5-20251001)
      - max_tokens: int (opcional, default: 500)
    
    Output (como en el nodo Return de n8n):
      - texto: str
      - tokens_input: int
      - tokens_output: int
      - coste_usd: float
    """
    kwargs = {
        "model": modelo,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": prompt}]
    }
    if system:
        kwargs["system"] = system
    
    resp = client.messages.create(**kwargs)
    
    p = PRECIOS_MODELOS.get(modelo, PRECIOS_MODELOS["claude-haiku-4-5-20251001"])
    coste = (
        resp.usage.input_tokens * p["input"] + 
        resp.usage.output_tokens * p["output"]
    ) / 1_000_000
    
    return {
        "texto": resp.content[0].text,
        "tokens_input": resp.usage.input_tokens,
        "tokens_output": resp.usage.output_tokens,
        "coste_usd": round(coste, 6)
    }

# Invocar el sub-workflow (como lo haría un nodo Execute Workflow en n8n)
resultado = sw_llamar_claude(
    prompt="Clasifica este email: Urgente: el servidor está caído",
    max_tokens=100
)

print("Resultado del sub-workflow SW_LlamarClaude:")
print(f"  Texto: {resultado['texto'][:100]}")
print(f"  Tokens: {resultado['tokens_input']} in / {resultado['tokens_output']} out")
print(f"  Coste: ${resultado['coste_usd']}")

In [ ]:
# Simular múltiples workflows usando el mismo sub-workflow
TOTAL_COSTE_USD = 0

def workflow_clasificar_email(email_subject: str, email_body: str) -> str:
    """Workflow principal que invoca SW_LlamarClaude."""
    global TOTAL_COSTE_USD
    resultado = sw_llamar_claude(
        prompt=f"Clasifica este email en una palabra (soporte/comercial/spam/otro):\nAsunto: {email_subject}\nMensaje: {email_body}",
        max_tokens=50
    )
    TOTAL_COSTE_USD += resultado["coste_usd"]
    return resultado["texto"]

def workflow_resumir_documento(texto: str) -> str:
    """Otro workflow que reutiliza el mismo sub-workflow."""
    global TOTAL_COSTE_USD
    resultado = sw_llamar_claude(
        prompt=f"Resume en 2 frases: {texto[:500]}",
        max_tokens=150
    )
    TOTAL_COSTE_USD += resultado["coste_usd"]
    return resultado["texto"]

# Los dos workflows comparten el mismo sub-workflow
cat = workflow_clasificar_email("Factura pendiente", "Adjuntamos la factura de abril")
resumen = workflow_resumir_documento("El proyecto avanzó bien esta semana. Se completaron 5 tareas y se cerraron 3 bugs.")

print(f"Clasificación: {cat.strip()}")
print(f"Resumen: {resumen}")
print(f"\nCoste total de ambos workflows: ${TOTAL_COSTE_USD:.6f}")

## 2. Manejo de errores: reintentos con backoff exponencial

Equivalente Python del patrón `procesarConReintentos()` del tutorial.

In [ ]:
def con_reintentos(operacion, max_reintentos: int = 3, espera_inicial: float = 1.0):
    """
    Wrapper de reintentos con backoff exponencial.
    En n8n se implementa con el campo 'Retry on Fail' del nodo HTTP Request.
    Para lógica más compleja, se usa el patrón del Code Node del tutorial.
    """
    ultimo_error = None
    
    for intento in range(1, max_reintentos + 1):
        try:
            print(f"  Intento {intento}/{max_reintentos}...")
            return operacion()
        except anthropic.RateLimitError as e:
            ultimo_error = e
            if intento < max_reintentos:
                espera = espera_inicial * intento
                print(f"  Rate limit. Esperando {espera:.1f}s...")
                time.sleep(espera)
        except anthropic.BadRequestError as e:
            # Error 4xx: no reintentar
            print(f"  Error de cliente (4xx): {e}. Sin reintentos.")
            raise
        except Exception as e:
            ultimo_error = e
            if intento < max_reintentos:
                espera = espera_inicial * intento
                print(f"  Error transitorio: {e}. Esperando {espera:.1f}s...")
                time.sleep(espera)
    
    raise ultimo_error

# Ejemplo de uso con la API de Claude
print("Ejecutando llamada con reintentos:")
resultado = con_reintentos(
    lambda: sw_llamar_claude("Responde 'OK' en una sola palabra.", max_tokens=10)
)
print(f"Resultado: {resultado['texto']}")

In [ ]:
# Simular fallo transitorio para demostrar los reintentos
CONTADOR_FALLOS = {"n": 0}

def operacion_con_fallos_transitorios():
    """Simula una operación que falla las primeras 2 veces."""
    CONTADOR_FALLOS["n"] += 1
    if CONTADOR_FALLOS["n"] < 3:
        raise ConnectionError(f"Timeout simulado (intento {CONTADOR_FALLOS['n']})")
    return {"texto": "Éxito tras reintentos", "tokens_input": 0, "tokens_output": 0, "coste_usd": 0}

print("Simulando fallos transitorios:")
try:
    resultado = con_reintentos(operacion_con_fallos_transitorios, max_reintentos=3, espera_inicial=0.1)
    print(f"\n✅ Éxito: {resultado['texto']}")
except Exception as e:
    print(f"\n❌ Falló definitivamente: {e}")

## 3. Rate Limiting con estado persistente

En n8n: `$getWorkflowStaticData('global')` persiste entre ejecuciones.
En Python: usamos un objeto de clase que simula ese estado global.

In [ ]:
class RateLimiterN8n:
    """
    Implementa el rate limiting del tutorial usando ventana deslizante.
    Equivalente al patrón $getWorkflowStaticData('global') de n8n.
    """
    
    def __init__(self, max_llamadas: int = 50, ventana_segundos: int = 60):
        self.max_llamadas = max_llamadas
        self.ventana_ms = ventana_segundos * 1000
        # En n8n: estatico.historialLlamadas = []
        self._historial = []
    
    def permitir(self) -> bool:
        """Verifica si se puede hacer una llamada. Retorna True si está permitido."""
        ahora = int(time.time() * 1000)  # milliseconds
        
        # Limpiar historial antiguo (fuera de la ventana)
        self._historial = [t for t in self._historial if ahora - t < self.ventana_ms]
        
        if len(self._historial) >= self.max_llamadas:
            espera_ms = self.ventana_ms - (ahora - self._historial[0])
            print(f"  ⚠️ Rate limit: espera {espera_ms/1000:.1f}s")
            return False
        
        self._historial.append(ahora)
        return True
    
    def stats(self) -> dict:
        return {
            "llamadas_en_ventana": len(self._historial),
            "max_llamadas": self.max_llamadas,
            "disponibles": self.max_llamadas - len(self._historial)
        }

def llamar_claude_con_rate_limit(rl: RateLimiterN8n, prompt: str) -> dict | None:
    """Equivalente al nodo Code con rate limiting en n8n."""
    if not rl.permitir():
        return None
    return sw_llamar_claude(prompt, max_tokens=50)

# Simular 5 llamadas en ráfaga
rl = RateLimiterN8n(max_llamadas=3, ventana_segundos=10)  # Límite bajo para demo

print("Simulando 5 llamadas con rate limit de 3/10s:")
for i in range(5):
    resultado = llamar_claude_con_rate_limit(rl, f"Di el número {i+1} y nada más.")
    stats = rl.stats()
    if resultado:
        print(f"  Llamada {i+1}: ✅ | En ventana: {stats['llamadas_en_ventana']}/{stats['max_llamadas']}")
    else:
        print(f"  Llamada {i+1}: 🚫 bloqueada | En ventana: {stats['llamadas_en_ventana']}/{stats['max_llamadas']}")

## 4. Workflow de errores centralizado

In [ ]:
LOG_ERRORES = []  # En n8n: Google Sheets donde se guardan los errores

def workflow_errores_centralizado(workflow_nombre: str, nodo: str, 
                                   error: Exception, execution_id: str) -> dict:
    """
    En n8n: Settings → Error Workflow de cada workflow apunta a este workflow.
    Se activa automáticamente cuando cualquier nodo lanza una excepción.
    """
    registro = {
        "workflow": workflow_nombre,
        "nodo": nodo,
        "error": str(error),
        "timestamp": datetime.now().isoformat(),
        "execution_id": execution_id
    }
    LOG_ERRORES.append(registro)
    
    # En n8n: HTTP Request → Slack webhook
    mensaje_slack = f"""❌ *Fallo en workflow de IA*
*Workflow:* {workflow_nombre}
*Nodo:* {nodo}
*Error:* {str(error)}
*Ejecución:* {execution_id}"""
    print(f"[SLACK #alertas-ia]\n{mensaje_slack}")
    
    # En n8n: Google Sheets → append row
    print(f"\n[SHEETS] Fila añadida al log de errores")
    
    return registro

def ejecutar_con_error_workflow(nombre: str, operacion):
    """Wrapper que captura errores y los envía al workflow centralizado."""
    try:
        return operacion()
    except Exception as e:
        return workflow_errores_centralizado(
            workflow_nombre=nombre,
            nodo="Code: llamar Claude",
            error=e,
            execution_id=f"exec_{int(time.time())}"
        )

# Simular un fallo en un workflow
print("Simulando fallo en workflow de clasificación:\n")

def operacion_que_falla():
    raise ValueError("Token API no configurado en variables de entorno")

resultado = ejecutar_con_error_workflow("Email-Clasificacion", operacion_que_falla)
print(f"\nTotal errores en log: {len(LOG_ERRORES)}")

## 5. Checklist de producción

In [ ]:
import os

CHECKLIST_PRODUCCION = {
    "Resiliencia": [
        ("error_workflow_configurado", True),
        ("reintentos_automaticos", True),
        ("timeout_definido", True),
        ("validacion_inputs_webhook", True),
    ],
    "Seguridad": [
        ("api_keys_en_env_vars", True),
        ("auth_basica_activada", True),
        ("webhook_secrets_validados", True),
        ("https_certificado_valido", True),
    ],
    "Monitorización": [
        ("alertas_error_slack", True),
        ("log_ejecuciones_bd", True),
        ("coste_por_workflow_registrado", False),  # Pendiente
    ],
    "Mantenimiento": [
        ("workflows_en_control_versiones", True),
        ("documentacion_workflows", False),  # Pendiente
        ("entorno_staging", True),
    ]
}

def evaluar_checklist(checklist: dict) -> dict:
    """Evalúa el checklist y genera un informe con Claude."""
    pendientes = []
    completados = []
    
    for categoria, items in checklist.items():
        for nombre, estado in items:
            if estado:
                completados.append(f"{categoria}: {nombre}")
            else:
                pendientes.append(f"{categoria}: {nombre}")
    
    puntuacion = len(completados) / (len(completados) + len(pendientes)) * 100
    
    # Generar recomendaciones con Claude
    if pendientes:
        resp = sw_llamar_claude(
            prompt=f"""Estos items de producción están pendientes en un sistema de workflows de IA:
{chr(10).join(f'- {p}' for p in pendientes)}

En 3 líneas: prioriza cuáles resolver primero y por qué. Sé directo.""",
            max_tokens=200
        )
        recomendacion = resp["texto"]
    else:
        recomendacion = "✅ Todos los checks superados. Sistema listo para producción."
    
    return {
        "puntuacion": round(puntuacion, 1),
        "completados": len(completados),
        "pendientes": len(pendientes),
        "items_pendientes": pendientes,
        "listo_produccion": puntuacion == 100,
        "recomendacion": recomendacion
    }

print("EVALUANDO CHECKLIST DE PRODUCCIÓN")
print("="*55)

for categoria, items in CHECKLIST_PRODUCCION.items():
    print(f"\n{categoria}:")
    for nombre, estado in items:
        icono = "✓" if estado else "✗"
        print(f"  [{icono}] {nombre}")

evaluacion = evaluar_checklist(CHECKLIST_PRODUCCION)
print(f"\n{'='*55}")
print(f"Puntuación: {evaluacion['puntuacion']}%")
print(f"Completados: {evaluacion['completados']} | Pendientes: {evaluacion['pendientes']}")
print(f"¿Listo para producción? {'✅ SÍ' if evaluacion['listo_produccion'] else '❌ NO'}")
print(f"\nRecomendación:\n{evaluacion['recomendacion']}")

## Resumen: Arquitectura de producción n8n + Claude

```
CAPA                  EN n8n                    EN PYTHON (este notebook)
─────────────────────────────────────────────────────────────────────────
Modularidad       Sub-workflows                sw_llamar_claude()
Resiliencia       Retry on Fail + Error WF     con_reintentos() + try/except
Rate limiting     $getWorkflowStaticData()     RateLimiterN8n class
Monitorización    Error Workflow → Slack        workflow_errores_centralizado()
Producción        Docker Compose + .env         os.environ + checklist
```

### Costes del sub-workflow SW_LlamarClaude

In [ ]:
# Proyección de costes usando el sub-workflow centralizado
print("PROYECCIÓN DE COSTES — Sub-workflow SW_LlamarClaude")
print("Modelo: claude-haiku-4-5-20251001")
print(f"{'─'*60}")
print(f"{'Ejecuciones/día':20} {'$/día':10} {'$/mes':10} {'$/año':10}")
print(f"{'─'*60}")

# Estimación: 500 tokens input + 300 tokens output por llamada media
coste_por_llamada = (500 * 0.80 + 300 * 4.00) / 1_000_000

for ejecuciones_dia in [50, 200, 500, 1000, 5000]:
    dia = coste_por_llamada * ejecuciones_dia
    mes = dia * 30
    año = mes * 12
    print(f"{ejecuciones_dia:<20} ${dia:<9.4f} ${mes:<9.2f} ${año:<.0f}")

print(f"\nNote: a 5.000 ejecuciones/día Sonnet costaría ~${coste_por_llamada * 5000 * 30 * (3.00/0.80 + 15.00/4.00)/2:.0f}/mes")